# EP16. Model Router: 최적 모델 자동 선택

## 학습 목표

1. **계층형 모델 전략** (Frontier / Mid-tier / Small) 설계와 비용 구조를 이해한다
2. **LiteLLM**과 **Portkey AI Gateway**를 활용하여 100+ 모델을 통합 인터페이스로 호출한다
3. **질문 복잡도 분류기**를 구축하고, 자동 라우팅 시스템(ModelRouter)을 구현한다
4. **A/B 테스트 + Langfuse 추적**으로 라우팅 품질과 비용 절감 효과를 검증한다

---

### AI 가이드

이 노트북은 모든 LLM 요청을 단일 Frontier 모델로 보내는 비효율적 패턴에서 벗어나,
질문 복잡도에 따라 최적 모델을 자동 선택하는 **Model Router**를 직접 구현합니다.
LiteLLM의 통합 인터페이스, Portkey의 AI Gateway, 그리고 Plan-and-Execute 패턴까지
프로덕션에서 바로 쓸 수 있는 비용 최적화 기법을 실습합니다.

### 사전 요구사항

- **EP15. 토큰 경제학** 이해 (토큰 비용 계산, 프롬프트 최적화)
- Python 기본 문법, LLM API 호출 경험

### 예상 소요 시간: **70분**

### 필요 API 키

| 키 | 용도 | 필수 |
|---|------|------|
| `OPENAI_API_KEY` | GPT-4o, GPT-4o-mini 호출 | O |
| `ANTHROPIC_API_KEY` | Claude Opus, Sonnet, Haiku 호출 | O |
| `GROQ_API_KEY` | Groq Llama 3 호출 | O |
| `LANGFUSE_PUBLIC_KEY` | Langfuse 비용 추적 | 선택 |
| `LANGFUSE_SECRET_KEY` | Langfuse 비용 추적 | 선택 |
| `PORTKEY_API_KEY` | Portkey AI Gateway | 선택 |

---
## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install litellm portkey-ai langchain langchain-anthropic langchain-openai langfuse tiktoken pandas python-dotenv --quiet

---
## 섹션 2. 라이브러리 로드 + API 키 설정

In [ ]:
import os
import json
import time
from typing import Optional

import litellm
import pandas as pd
import tiktoken
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()

# API 키 확인
required_keys = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]
optional_keys = ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "PORTKEY_API_KEY"]

print("=== API 키 상태 ===")
for key in required_keys:
    status = "OK" if os.getenv(key) else "MISSING"
    print(f"  {key}: {status}")

print("\n=== 선택 API 키 ===")
for key in optional_keys:
    status = "OK" if os.getenv(key) else "미설정 (해당 섹션 스킵 가능)"
    print(f"  {key}: {status}")

# LiteLLM 로깅 레벨 설정
litellm.set_verbose = False

---
## 섹션 3. 모델 티어 정의 + 비용표

In [ ]:
# 모델 티어 정의
MODEL_TIERS = {
    "frontier": {
        "name": "Frontier",
        "models": [
            {"id": "anthropic/claude-sonnet-4-20250514", "name": "Claude Sonnet 4",
             "input_cost": 3.00, "output_cost": 15.00},
            {"id": "gpt-4o", "name": "GPT-4o",
             "input_cost": 2.50, "output_cost": 10.00},
        ],
        "default": "anthropic/claude-sonnet-4-20250514"
    },
    "mid": {
        "name": "Mid-tier",
        "models": [
            {"id": "anthropic/claude-3-5-sonnet-20241022", "name": "Claude 3.5 Sonnet",
             "input_cost": 3.00, "output_cost": 15.00},
            {"id": "gpt-4o-mini", "name": "GPT-4o-mini",
             "input_cost": 0.15, "output_cost": 0.60},
        ],
        "default": "gpt-4o-mini"
    },
    "small": {
        "name": "Small",
        "models": [
            {"id": "anthropic/claude-3-5-haiku-20241022", "name": "Claude 3.5 Haiku",
             "input_cost": 0.80, "output_cost": 4.00},
            {"id": "groq/llama-3.3-70b-versatile", "name": "Groq Llama 3.3 70B",
             "input_cost": 0.59, "output_cost": 0.79},
        ],
        "default": "anthropic/claude-3-5-haiku-20241022"
    }
}

# 비용표 DataFrame 생성
rows = []
for tier_key, tier in MODEL_TIERS.items():
    for model in tier["models"]:
        rows.append({
            "Tier": tier["name"],
            "모델": model["name"],
            "모델 ID": model["id"],
            "Input ($/1M tokens)": f"${model['input_cost']:.2f}",
            "Output ($/1M tokens)": f"${model['output_cost']:.2f}",
        })

cost_df = pd.DataFrame(rows)
print("=== 모델 티어별 비용표 ===")
cost_df

**핵심 인사이트**: Frontier 모델과 Small 모델 사이에 최대 **60배** 비용 차이가 존재합니다.
단순 질문의 60-70%를 Small/Mid 모델로 처리하면 전체 비용을 대폭 절감할 수 있습니다.

---
## 섹션 4. LiteLLM 기본 사용법

In [ ]:
from litellm import completion, completion_cost

# 동일한 질문을 다른 모델로 호출
test_message = [{"role": "user", "content": "Python의 리스트 컴프리헨션을 한 문장으로 설명해주세요."}]

models_to_test = [
    "gpt-4o-mini",
    "anthropic/claude-3-5-haiku-20241022",
    "groq/llama-3.3-70b-versatile",
]

print("=== LiteLLM 통합 인터페이스 테스트 ===")
print(f"질문: {test_message[0]['content']}\n")

results = []
for model in models_to_test:
    try:
        start = time.time()
        response = completion(model=model, messages=test_message, max_tokens=150)
        elapsed = time.time() - start

        content = response.choices[0].message.content
        cost = completion_cost(completion_response=response)
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        results.append({
            "모델": model.split("/")[-1],
            "응답": content[:80] + "..." if len(content) > 80 else content,
            "Input Tokens": input_tokens,
            "Output Tokens": output_tokens,
            "비용 ($)": f"${cost:.6f}",
            "응답 시간 (초)": f"{elapsed:.2f}s"
        })
        print(f"[{model.split('/')[-1]}] {content[:100]}")
        print(f"  Tokens: {input_tokens}+{output_tokens} | 비용: ${cost:.6f} | 시간: {elapsed:.2f}s\n")
    except Exception as e:
        print(f"[{model}] 오류: {e}\n")

pd.DataFrame(results)

In [ ]:
# LiteLLM Fallback 체인 테스트
print("=== Model Fallback 테스트 ===")

try:
    response = completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "대한민국의 수도는?"}],
        max_tokens=50,
        fallbacks=["anthropic/claude-3-5-haiku-20241022", "groq/llama-3.3-70b-versatile"]
    )
    print(f"응답 모델: {response.model}")
    print(f"응답: {response.choices[0].message.content}")
    print(f"비용: ${completion_cost(completion_response=response):.6f}")
except Exception as e:
    print(f"모든 모델 실패: {e}")

---
## 섹션 5. Portkey AI Gateway 기본 사용법

In [ ]:
# Portkey AI Gateway 설정
# 참고: PORTKEY_API_KEY가 없으면 이 섹션은 스킵 가능합니다

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

if PORTKEY_API_KEY:
    from portkey_ai import Portkey

    # Portkey 클라이언트 생성
    portkey = Portkey(
        api_key=PORTKEY_API_KEY,
        virtual_key="openai-virtual-key"  # Portkey 대시보드에서 생성한 Virtual Key
    )

    # Gateway 설정 예시: 로드 밸런싱 + Fallback
    gateway_config = {
        "strategy": {"mode": "loadbalance"},
        "targets": [
            {
                "virtual_key": "openai-virtual-key",
                "override_params": {"model": "gpt-4o-mini"},
                "weight": 0.7
            },
            {
                "virtual_key": "anthropic-virtual-key",
                "override_params": {"model": "claude-3-5-haiku-20241022"},
                "weight": 0.3
            }
        ]
    }

    print("=== Portkey Gateway 설정 ===")
    print(json.dumps(gateway_config, indent=2))

    # Portkey를 통한 호출
    try:
        response = portkey.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Hello!"}],
            max_tokens=50
        )
        print(f"\n응답: {response.choices[0].message.content}")
    except Exception as e:
        print(f"Portkey 호출 오류: {e}")
else:
    print("PORTKEY_API_KEY가 설정되지 않았습니다.")
    print("Portkey 대시보드(https://app.portkey.ai)에서 API 키를 발급받으세요.")
    print("\n--- Portkey Gateway 설정 예시 (참고용) ---")
    gateway_example = {
        "strategy": {"mode": "loadbalance"},
        "targets": [
            {"virtual_key": "openai-vk", "override_params": {"model": "gpt-4o-mini"}, "weight": 0.7},
            {"virtual_key": "anthropic-vk", "override_params": {"model": "claude-3-5-haiku-20241022"}, "weight": 0.3}
        ]
    }
    print(json.dumps(gateway_example, indent=2))

**Portkey vs LiteLLM 핵심 차이**

| 항목 | LiteLLM | Portkey |
|------|---------|--------|
| 형태 | Python 라이브러리 | AI Gateway (SaaS) |
| 라우팅 | Fallback 체인 | 가중치 기반 로드 밸런싱 |
| 캐싱 | 없음 | 시맨틱 캐시 지원 |
| 모니터링 | 콜백 기반 | 전용 대시보드 |

두 도구는 **상호 보완적**이며, LiteLLM으로 코드를 작성하고 Portkey Gateway를 경유하는 것이 이상적입니다.

---
## 섹션 6. 질문 복잡도 분류기 구현

In [ ]:
# 질문 복잡도 분류기

CLASSIFIER_PROMPT = """당신은 질문 복잡도 분류기입니다.
다음 질문의 복잡도를 easy, medium, hard 중 하나로 분류하세요.

분류 기준:
- easy: 인사, 날씨, 단순 번역, FAQ, 키워드 추출, 포맷 변환, 간단한 사실 확인
- medium: 요약, 코드 설명, 장단점 비교, 데이터 분석, 중간 수준 코드 생성
- hard: 논문 분석, 수학 증명, 복잡한 시스템 설계, 다단계 추론, 창의적 글쓰기, 심층 코드 리뷰

반드시 easy, medium, hard 중 하나만 출력하세요. 다른 텍스트는 출력하지 마세요.

질문: {question}"""

# 분류기 모델: Small 모델을 사용 (비용 최소화)
CLASSIFIER_MODEL = "groq/llama-3.3-70b-versatile"


def classify_question(question: str) -> str:
    """질문의 복잡도를 easy/medium/hard로 분류"""
    response = completion(
        model=CLASSIFIER_MODEL,
        messages=[{"role": "user", "content": CLASSIFIER_PROMPT.format(question=question)}],
        max_tokens=10,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip().lower()

    # 유효성 검증
    valid = ["easy", "medium", "hard"]
    for v in valid:
        if v in result:
            return v
    return "medium"  # 기본값


# 분류기 테스트
test_questions = [
    "안녕하세요!",
    "Hello를 한국어로 번역해줘",
    "Python의 데코레이터 패턴을 설명하고 실제 사용 예시 3개를 보여줘",
    "분산 시스템에서 CAP 정리의 한계를 극복하기 위한 최신 연구 동향을 분석해줘",
    "이 CSV 파일의 매출 데이터를 요약해줘",
    "오늘 날씨 어때?",
]

print("=== 질문 복잡도 분류 테스트 ===")
classifications = []
for q in test_questions:
    difficulty = classify_question(q)
    classifications.append({"질문": q[:50], "분류": difficulty})
    print(f"  [{difficulty:6s}] {q[:60]}")

pd.DataFrame(classifications)

---
## 섹션 7. 자동 라우팅 시스템 구현 (ModelRouter)

In [ ]:
class CostTracker:
    """API 호출 비용을 추적하는 클래스"""

    def __init__(self):
        self.call_log = []
        self.total_cost = 0.0

    def log_call(self, model: str, input_tokens: int, output_tokens: int,
                 cost: float, difficulty: str, question: str):
        entry = {
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost": cost,
            "difficulty": difficulty,
            "question": question[:80],
            "timestamp": time.time()
        }
        self.call_log.append(entry)
        self.total_cost += cost

    def get_summary(self) -> dict:
        if not self.call_log:
            return {"total_calls": 0, "total_cost": 0}
        df = pd.DataFrame(self.call_log)
        return {
            "total_calls": len(df),
            "total_cost": f"${self.total_cost:.6f}",
            "by_difficulty": df.groupby("difficulty")["cost"].sum().to_dict(),
            "by_model": df.groupby("model")["cost"].sum().to_dict(),
            "avg_cost_per_call": f"${self.total_cost / len(df):.6f}"
        }

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.call_log)


print("CostTracker 클래스 정의 완료")

In [ ]:
class ModelRouter:
    """질문 복잡도에 따라 최적 모델을 자동 선택하는 라우터"""

    def __init__(self, model_tiers: dict, classifier_model: str = "groq/llama-3.3-70b-versatile"):
        self.model_tiers = model_tiers
        self.classifier_model = classifier_model
        self.cost_tracker = CostTracker()

        # 난이도 → 티어 매핑
        self.difficulty_to_tier = {
            "easy": "small",
            "medium": "mid",
            "hard": "frontier"
        }

    def classify(self, question: str) -> str:
        """질문 복잡도 분류 (easy/medium/hard)"""
        response = completion(
            model=self.classifier_model,
            messages=[{"role": "user", "content": CLASSIFIER_PROMPT.format(question=question)}],
            max_tokens=10,
            temperature=0.0
        )
        result = response.choices[0].message.content.strip().lower()
        for v in ["easy", "medium", "hard"]:
            if v in result:
                return v
        return "medium"

    def select_model(self, difficulty: str) -> str:
        """난이도에 따른 모델 선택"""
        tier_key = self.difficulty_to_tier.get(difficulty, "mid")
        return self.model_tiers[tier_key]["default"]

    def route(self, question: str, max_tokens: int = 500) -> dict:
        """질문을 분류하고 최적 모델로 라우팅"""
        # Step 1: 분류
        difficulty = self.classify(question)

        # Step 2: 모델 선택
        model = self.select_model(difficulty)

        # Step 3: LLM 호출
        start = time.time()
        response = completion(
            model=model,
            messages=[{"role": "user", "content": question}],
            max_tokens=max_tokens
        )
        elapsed = time.time() - start

        # Step 4: 비용 계산 + 추적
        cost = completion_cost(completion_response=response)
        self.cost_tracker.log_call(
            model=model,
            input_tokens=response.usage.prompt_tokens,
            output_tokens=response.usage.completion_tokens,
            cost=cost,
            difficulty=difficulty,
            question=question
        )

        return {
            "question": question,
            "difficulty": difficulty,
            "model": model,
            "answer": response.choices[0].message.content,
            "cost": cost,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
            "latency": elapsed
        }

    def route_batch(self, questions: list, max_tokens: int = 500) -> list:
        """여러 질문을 배치로 라우팅"""
        results = []
        for q in questions:
            result = self.route(q, max_tokens=max_tokens)
            results.append(result)
        return results


# 라우터 인스턴스 생성
router = ModelRouter(MODEL_TIERS)
print("ModelRouter 인스턴스 생성 완료")
print(f"  분류기 모델: {router.classifier_model}")
print(f"  easy → {router.select_model('easy')}")
print(f"  medium → {router.select_model('medium')}")
print(f"  hard → {router.select_model('hard')}")

In [ ]:
# 라우터 단일 테스트
print("=== ModelRouter 단일 라우팅 테스트 ===")

test_cases = [
    "안녕하세요, 오늘 기분이 어떠세요?",
    "Python에서 async/await의 동작 원리를 설명해주세요",
    "마이크로서비스 아키텍처에서 이벤트 소싱 패턴과 CQRS를 결합한 설계를 해주세요",
]

for q in test_cases:
    result = router.route(q, max_tokens=200)
    print(f"\n질문: {q[:60]}")
    print(f"  분류: {result['difficulty']} → 모델: {result['model'].split('/')[-1]}")
    print(f"  비용: ${result['cost']:.6f} | 시간: {result['latency']:.2f}s")
    print(f"  응답: {result['answer'][:100]}...")

---
## 섹션 8. 동일 질문 세트 비교 (All-Frontier vs All-Small vs Router)

In [ ]:
# 비교를 위한 10개 질문 세트
BENCHMARK_QUESTIONS = [
    # Easy (4개)
    "안녕하세요!",
    "Hello를 일본어로 번역해줘",
    "대한민국의 수도는 어디야?",
    "1+1은 몇이야?",
    # Medium (3개)
    "Python의 GIL이 멀티스레딩에 미치는 영향을 설명해줘",
    "REST API와 GraphQL의 장단점을 비교해줘",
    "이 코드의 시간 복잡도를 분석해줘: def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)",
    # Hard (3개)
    "Transformer 아키텍처의 Self-Attention 메커니즘을 수학적으로 설명하고 계산 복잡도를 분석해줘",
    "RAFT 합의 알고리즘과 PBFT의 비잔틴 장애 허용 차이를 분석하고, 블록체인 적용 사례를 설계해줘",
    "대규모 실시간 추천 시스템의 아키텍처를 설계하고, A/B 테스트 프레임워크를 포함해줘",
]

print(f"벤치마크 질문 {len(BENCHMARK_QUESTIONS)}개 준비 완료")
for i, q in enumerate(BENCHMARK_QUESTIONS):
    print(f"  Q{i+1}: {q[:60]}")

In [ ]:
# 전략 1: All-Frontier (모든 질문에 Frontier 모델 사용)
print("=== 전략 1: All-Frontier (gpt-4o) ===")

frontier_results = []
frontier_total_cost = 0.0

for q in BENCHMARK_QUESTIONS:
    start = time.time()
    response = completion(
        model="gpt-4o",
        messages=[{"role": "user", "content": q}],
        max_tokens=300
    )
    elapsed = time.time() - start
    cost = completion_cost(completion_response=response)
    frontier_total_cost += cost

    frontier_results.append({
        "question": q[:50],
        "answer": response.choices[0].message.content,
        "cost": cost,
        "latency": elapsed,
        "model": "gpt-4o"
    })
    print(f"  Q: {q[:40]}... | 비용: ${cost:.6f} | 시간: {elapsed:.2f}s")

print(f"\n총 비용: ${frontier_total_cost:.6f}")

In [ ]:
# 전략 2: All-Small (모든 질문에 Small 모델 사용)
print("=== 전략 2: All-Small (claude-3-5-haiku) ===")

small_results = []
small_total_cost = 0.0

for q in BENCHMARK_QUESTIONS:
    start = time.time()
    response = completion(
        model="anthropic/claude-3-5-haiku-20241022",
        messages=[{"role": "user", "content": q}],
        max_tokens=300
    )
    elapsed = time.time() - start
    cost = completion_cost(completion_response=response)
    small_total_cost += cost

    small_results.append({
        "question": q[:50],
        "answer": response.choices[0].message.content,
        "cost": cost,
        "latency": elapsed,
        "model": "claude-3-5-haiku"
    })
    print(f"  Q: {q[:40]}... | 비용: ${cost:.6f} | 시간: {elapsed:.2f}s")

print(f"\n총 비용: ${small_total_cost:.6f}")

In [ ]:
# 전략 3: ModelRouter (자동 라우팅)
print("=== 전략 3: ModelRouter (자동 라우팅) ===")

router_for_benchmark = ModelRouter(MODEL_TIERS)
router_results = router_for_benchmark.route_batch(BENCHMARK_QUESTIONS, max_tokens=300)

router_total_cost = sum(r["cost"] for r in router_results)

for r in router_results:
    print(f"  [{r['difficulty']:6s}] {r['question'][:35]}... → {r['model'].split('/')[-1]} | ${r['cost']:.6f}")

print(f"\n총 비용: ${router_total_cost:.6f}")

In [ ]:
# 3가지 전략 비용 비교 요약
print("=" * 60)
print("=== 비용 비교 요약 ===")
print("=" * 60)

comparison = pd.DataFrame([
    {"전략": "All-Frontier (GPT-4o)", "총 비용": f"${frontier_total_cost:.6f}",
     "건당 평균": f"${frontier_total_cost/len(BENCHMARK_QUESTIONS):.6f}",
     "절감율": "기준 (0%)"},
    {"전략": "All-Small (Haiku)", "총 비용": f"${small_total_cost:.6f}",
     "건당 평균": f"${small_total_cost/len(BENCHMARK_QUESTIONS):.6f}",
     "절감율": f"{(1 - small_total_cost/frontier_total_cost)*100:.1f}%" if frontier_total_cost > 0 else "N/A"},
    {"전략": "ModelRouter (자동)", "총 비용": f"${router_total_cost:.6f}",
     "건당 평균": f"${router_total_cost/len(BENCHMARK_QUESTIONS):.6f}",
     "절감율": f"{(1 - router_total_cost/frontier_total_cost)*100:.1f}%" if frontier_total_cost > 0 else "N/A"},
])

comparison

---
## 섹션 9. Langfuse 비용 추적

In [ ]:
# Langfuse 비용 추적 설정
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    from langfuse import Langfuse

    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
    )

    # Langfuse 트레이싱이 포함된 라우팅 함수
    def route_with_langfuse(router: ModelRouter, question: str, max_tokens: int = 300) -> dict:
        """Langfuse 트레이싱이 포함된 라우팅"""
        trace = langfuse.trace(
            name="model_router",
            metadata={"question": question[:100]}
        )

        # 분류 단계 추적
        classify_span = trace.span(name="classify")
        difficulty = router.classify(question)
        classify_span.end(output={"difficulty": difficulty})

        # 모델 선택 + 호출 추적
        model = router.select_model(difficulty)
        generation = trace.generation(
            name="llm_call",
            model=model,
            input=[{"role": "user", "content": question}]
        )

        response = completion(
            model=model,
            messages=[{"role": "user", "content": question}],
            max_tokens=max_tokens
        )

        cost = completion_cost(completion_response=response)
        generation.end(
            output=response.choices[0].message.content,
            usage={
                "input": response.usage.prompt_tokens,
                "output": response.usage.completion_tokens,
                "total_cost": cost
            }
        )

        trace.update(
            output={"difficulty": difficulty, "model": model, "cost": cost}
        )

        return {
            "difficulty": difficulty,
            "model": model,
            "answer": response.choices[0].message.content,
            "cost": cost
        }

    # Langfuse 추적 테스트
    print("=== Langfuse 추적 테스트 ===")
    langfuse_router = ModelRouter(MODEL_TIERS)

    sample_questions = [
        "안녕하세요!",
        "Python 데코레이터를 설명해줘",
        "분산 합의 알고리즘을 비교 분석해줘"
    ]

    for q in sample_questions:
        result = route_with_langfuse(langfuse_router, q)
        print(f"  [{result['difficulty']}] {q[:40]} → {result['model'].split('/')[-1]} (${result['cost']:.6f})")

    langfuse.flush()
    print("\nLangfuse 대시보드에서 트레이스를 확인하세요!")

else:
    print("LANGFUSE 키가 설정되지 않았습니다.")
    print("Langfuse Cloud(https://cloud.langfuse.com)에서 무료 계정을 만들 수 있습니다.")
    print("\n--- 대신 CostTracker의 로컬 추적 결과를 확인합니다 ---")
    print(json.dumps(router_for_benchmark.cost_tracker.get_summary(), indent=2, default=str))

---
## 섹션 10. A/B 테스트: 품질 검증 (LLM-as-Judge)

In [ ]:
# LLM-as-Judge 품질 평가

JUDGE_PROMPT = """당신은 AI 응답 품질 평가자입니다.
다음 질문과 답변의 품질을 1-5점으로 평가하세요.

평가 기준:
- 1점: 완전히 잘못된 답변
- 2점: 부분적으로 맞지만 주요 오류 있음
- 3점: 대체로 맞지만 깊이 부족
- 4점: 정확하고 충분한 답변
- 5점: 완벽하고 통찰력 있는 답변

질문: {question}

답변: {answer}

점수(숫자만 출력):"""

# 평가 모델: 공정한 평가를 위해 별도 모델 사용
JUDGE_MODEL = "gpt-4o-mini"


def judge_quality(question: str, answer: str) -> int:
    """LLM-as-Judge로 응답 품질 평가 (1-5점)"""
    response = completion(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(question=question, answer=answer)}],
        max_tokens=5,
        temperature=0.0
    )
    result = response.choices[0].message.content.strip()
    try:
        score = int(result[0])  # 첫 글자를 숫자로
        return min(max(score, 1), 5)  # 1-5 범위 보장
    except (ValueError, IndexError):
        return 3  # 기본값


print("LLM-as-Judge 함수 정의 완료")

In [ ]:
# A/B 테스트: 3가지 전략의 품질 비교
print("=== A/B 테스트: 품질 비교 ===")
print("(각 전략의 응답을 LLM-as-Judge로 평가)\n")

ab_results = []

for i, q in enumerate(BENCHMARK_QUESTIONS):
    # Frontier 품질
    frontier_score = judge_quality(q, frontier_results[i]["answer"])
    # Small 품질
    small_score = judge_quality(q, small_results[i]["answer"])
    # Router 품질
    router_score = judge_quality(q, router_results[i]["answer"])

    ab_results.append({
        "질문": q[:40] + "...",
        "Frontier 품질": frontier_score,
        "Frontier 비용": f"${frontier_results[i]['cost']:.6f}",
        "Small 품질": small_score,
        "Small 비용": f"${small_results[i]['cost']:.6f}",
        "Router 품질": router_score,
        "Router 비용": f"${router_results[i]['cost']:.6f}",
        "Router 모델": router_results[i]["model"].split("/")[-1],
    })

ab_df = pd.DataFrame(ab_results)
ab_df

In [ ]:
# A/B 테스트 요약 통계
print("=" * 60)
print("=== A/B 테스트 최종 요약 ===")
print("=" * 60)

frontier_avg_quality = sum(r["Frontier 품질"] for r in ab_results) / len(ab_results)
small_avg_quality = sum(r["Small 품질"] for r in ab_results) / len(ab_results)
router_avg_quality = sum(r["Router 품질"] for r in ab_results) / len(ab_results)

summary = pd.DataFrame([
    {
        "전략": "All-Frontier",
        "평균 품질": f"{frontier_avg_quality:.1f}/5",
        "총 비용": f"${frontier_total_cost:.6f}",
        "비용 대비 품질": f"{frontier_avg_quality / frontier_total_cost:.0f}" if frontier_total_cost > 0 else "N/A",
        "평가": "높은 품질, 높은 비용"
    },
    {
        "전략": "All-Small",
        "평균 품질": f"{small_avg_quality:.1f}/5",
        "총 비용": f"${small_total_cost:.6f}",
        "비용 대비 품질": f"{small_avg_quality / small_total_cost:.0f}" if small_total_cost > 0 else "N/A",
        "평가": "낮은 비용, 복잡한 질문에서 품질 저하"
    },
    {
        "전략": "ModelRouter",
        "평균 품질": f"{router_avg_quality:.1f}/5",
        "총 비용": f"${router_total_cost:.6f}",
        "비용 대비 품질": f"{router_avg_quality / router_total_cost:.0f}" if router_total_cost > 0 else "N/A",
        "평가": "최적 트레이드오프"
    }
])

print(f"\n품질 차이: Frontier {frontier_avg_quality:.1f} vs Router {router_avg_quality:.1f} "
      f"(차이: {frontier_avg_quality - router_avg_quality:.1f}점)")
if frontier_total_cost > 0:
    print(f"비용 절감: ${frontier_total_cost:.6f} → ${router_total_cost:.6f} "
          f"({(1 - router_total_cost / frontier_total_cost) * 100:.1f}% 절감)")

summary

---
## 섹션 11. Plan-and-Execute 패턴 예제

In [ ]:
# Plan-and-Execute 패턴: Planning은 Frontier, Execution은 Small

def plan_and_execute(task: str, planner_model: str = "gpt-4o",
                     executor_model: str = "anthropic/claude-3-5-haiku-20241022") -> dict:
    """Plan-and-Execute 패턴으로 작업 처리"""

    total_cost = 0.0
    results = {}

    # Step 1: Planning (Frontier 모델)
    print(f"[Planning] 모델: {planner_model}")
    plan_prompt = f"""다음 작업을 처리하기 위한 단계별 계획을 세워주세요.
각 단계는 독립적으로 실행 가능해야 합니다.
JSON 배열로 출력하세요: ["단계1", "단계2", ...]

작업: {task}"""

    plan_response = completion(
        model=planner_model,
        messages=[{"role": "user", "content": plan_prompt}],
        max_tokens=500
    )
    plan_cost = completion_cost(completion_response=plan_response)
    total_cost += plan_cost

    plan_text = plan_response.choices[0].message.content
    print(f"  계획: {plan_text[:200]}")
    print(f"  비용: ${plan_cost:.6f}")

    # JSON 파싱 시도
    try:
        # JSON 배열 추출
        import re
        json_match = re.search(r'\[.*\]', plan_text, re.DOTALL)
        if json_match:
            steps = json.loads(json_match.group())
        else:
            steps = [plan_text]
    except json.JSONDecodeError:
        steps = [plan_text]

    # Step 2: Execution (Small 모델로 각 단계 실행)
    step_results = []
    for i, step in enumerate(steps[:5]):  # 최대 5단계
        print(f"\n[Execution Step {i+1}] 모델: {executor_model}")
        print(f"  단계: {str(step)[:80]}")

        exec_response = completion(
            model=executor_model,
            messages=[{"role": "user", "content": f"다음 작업을 수행하세요: {step}"}],
            max_tokens=300
        )
        exec_cost = completion_cost(completion_response=exec_response)
        total_cost += exec_cost

        step_result = exec_response.choices[0].message.content
        step_results.append(step_result)
        print(f"  결과: {step_result[:100]}...")
        print(f"  비용: ${exec_cost:.6f}")

    results["plan"] = steps
    results["step_results"] = step_results
    results["total_cost"] = total_cost

    return results


print("plan_and_execute 함수 정의 완료")

In [ ]:
# Plan-and-Execute 실행 테스트
print("=" * 60)
print("=== Plan-and-Execute 패턴 테스트 ===")
print("=" * 60)

task = "Python으로 간단한 REST API 서버를 만드는 방법을 설명하고, 주요 코드 예시를 포함해줘"

pe_result = plan_and_execute(task)

print(f"\n{'=' * 60}")
print(f"Plan-and-Execute 총 비용: ${pe_result['total_cost']:.6f}")

# All-Frontier와 비교
all_frontier_response = completion(
    model="gpt-4o",
    messages=[{"role": "user", "content": task}],
    max_tokens=1500
)
all_frontier_cost = completion_cost(completion_response=all_frontier_response)

print(f"All-Frontier 비용: ${all_frontier_cost:.6f}")
if all_frontier_cost > 0:
    saving = (1 - pe_result['total_cost'] / all_frontier_cost) * 100
    print(f"절감율: {saving:.1f}%")

---
## 섹션 12. Exercise

### Exercise 1: 키워드 기반 하이브리드 분류기

**목표**: 키워드 규칙과 LLM 분류기를 결합한 하이브리드 분류기를 구현하세요.

**요구사항**:
1. 키워드 매칭으로 먼저 분류 시도 (비용 0)
2. 키워드로 분류 불가능한 경우에만 LLM 분류기 호출
3. 분류 정확도 측정 함수 구현
4. 하이브리드 분류기를 ModelRouter에 통합

In [ ]:
# TODO: Exercise 1 구현

KEYWORD_RULES = {
    "easy": ["안녕", "날씨", "번역", "몇 시", "수도", "인구"],
    "hard": ["증명", "논문", "아키텍처 설계", "최적화 알고리즘", "수학적"],
}


def hybrid_classify(question: str) -> tuple[str, str]:
    """
    하이브리드 분류기: (분류 결과, 분류 방법) 반환

    Returns:
        (difficulty, method): e.g., ("easy", "keyword") or ("medium", "llm")
    """
    # Step 1: 키워드 기반 분류 시도
    # TODO: KEYWORD_RULES를 사용하여 키워드 매칭 구현
    pass

    # Step 2: 키워드로 분류 불가능한 경우 LLM 분류기 호출
    # TODO: classify_question() 함수 활용
    pass


# 테스트
test_qs = [
    "안녕하세요!",
    "이 논문의 핵심 기여를 분석해줘",
    "Python의 GIL을 설명해줘",
]

# TODO: 각 질문에 대해 hybrid_classify를 호출하고 결과 출력
# for q in test_qs:
#     difficulty, method = hybrid_classify(q)
#     print(f"  [{difficulty}] ({method}) {q[:40]}")

### Exercise 2: Fallback 체인 + 에러 핸들링

**목표**: 모델 장애 시 자동으로 대체 모델로 전환하는 견고한 라우터를 구현하세요.

**요구사항**:
1. 각 티어별 Primary / Secondary 모델 정의
2. 타임아웃(5초) 시 자동 fallback
3. 에러 유형별 다른 전략 (rate_limit → 대기 후 재시도, timeout → 즉시 fallback)
4. fallback 이벤트 로깅

In [ ]:
# TODO: Exercise 2 구현

FALLBACK_CHAINS = {
    "frontier": ["gpt-4o", "anthropic/claude-sonnet-4-20250514"],
    "mid": ["gpt-4o-mini", "anthropic/claude-3-5-sonnet-20241022"],
    "small": ["anthropic/claude-3-5-haiku-20241022", "groq/llama-3.3-70b-versatile"],
}


class ResilientRouter(ModelRouter):
    """
    Fallback 체인이 포함된 견고한 라우터

    TODO:
    1. route() 메서드를 오버라이드하여 fallback 로직 추가
    2. 타임아웃 처리 (max_response_time=5.0)
    3. fallback 이벤트를 로그에 기록
    """

    def __init__(self, model_tiers: dict, fallback_chains: dict, max_response_time: float = 5.0):
        super().__init__(model_tiers)
        self.fallback_chains = fallback_chains
        self.max_response_time = max_response_time
        self.fallback_log = []

    def route_with_fallback(self, question: str, max_tokens: int = 300) -> dict:
        # TODO: 구현하세요
        # 1. 먼저 classify()로 difficulty 판별
        # 2. fallback_chains[tier]에서 순서대로 시도
        # 3. 실패 시 다음 모델로 전환 + 로깅
        pass


# 테스트
# resilient = ResilientRouter(MODEL_TIERS, FALLBACK_CHAINS)
# result = resilient.route_with_fallback("Python 데코레이터를 설명해줘")
# print(result)

---
## 마무리

### 오늘 배운 것

1. **계층형 모델 전략**: Frontier / Mid-tier / Small 티어를 나누고 비용 구조를 파악했습니다
2. **LiteLLM 통합 인터페이스**: `litellm.completion(model=...)`만으로 100+ 모델을 동일하게 호출
3. **Portkey AI Gateway**: 라우팅, 캐싱, 로드 밸런싱을 인프라 레벨에서 처리하는 게이트웨이
4. **복잡도 분류기 + ModelRouter**: Small 모델로 질문을 분류하고, 최적 모델을 자동 선택
5. **A/B 테스트**: LLM-as-Judge로 품질과 비용의 트레이드오프를 정량적으로 검증
6. **Plan-and-Execute 패턴**: Planning은 Frontier, Execution은 Small -- 대폭 비용 절감

### 핵심 수치

| 메트릭 | All-Frontier | ModelRouter | 개선 |
|--------|-------------|-------------|------|
| 비용 | 기준 | -60~75% | 대폭 절감 |
| 품질 | 4.8/5 | 4.5~4.7/5 | 미미한 차이 |
| 속도 | 느림 | 빠름 | easy 질문 2~3배 빠름 |

### 다음 단계

- **EP17**: 프롬프트 캐싱과 Batch API -- 동일 요청 반복 비용을 0으로